## Known from shared experimental data


| Category                         | Source                                  | Variables / Parameters                                       |
| -------------------------------- | --------------------------------------- | ------------------------------------------------------------ |
| **Time points**                  | AMBR CSVs (`Time`)                      | `t_eval` (can be extracted directly, in hours)               |
| **Observable signals**           | AMBR CSVs                               | DO (%), pH, OUR, CER, gas flows, OD (Optical Density)        |
| **Temperature**                  | AMBR CSVs / metadata (`ambr.xlsx`)      | `T_celsius` (usually 30 °C)                                  |
| **Liquid volume (approx.)**      | `ambr.xlsx` (`LiquidVolume`)            | `V_l` ≈ 0.2 L                                                |
| **Gas phase volume**             | fixed by model                          | `V_g` = 0.105 L (no need to fit)                             |
| **Feed / base / gas flow rates** | some runs have `Feed# Flow`, `Air flow` | can be read automatically; otherwise → estimated (see below) |


## Missing or incomplete experimentally: must be estimated or seeded

| Type                       | Parameter / Variable                                                            | Comment                                                                                                              |
| -------------------------- | ------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| **Initial conditions**     | `Biomass_0`                                                                     | OD₀ exists, but no OD→gDW/L calibration → conversion factor **must be estimated or assumed** (default 0.4 gDW/L/OD). |
|                            | `Sub_0`                                                                         | Initial substrate concentration absent → seeded **10 g/L** (must be verified or estimated).                          |
|                            | `O2_l_0`, `CO2_l_0`                                                             | Dissolved gas concentrations not recorded → **estimated from air equilibrium**.                                      |
|                            | Trace ions (`Ca`, `Mg`, `Cl`, etc.)                                             | Not measured → **seeded 0 mol/L**, it can be leaved fixed or estimated roughly from medium composition.                  |
| **Kinetic parameters**     | `mu_max`, `K_subs`                                                              | Not measurable directly → must be **estimated from Biomass/Sub curves**.                                             |
| **Gas transfer**           | `K_L_a`                                                                         | No experimental kLa values in data → **estimated from DO/gas dynamics**.                                             |
| **Feed composition**       | `C_Sub_f`, `C_H_a`, `C_Na_b`, etc.                                              | Feed concentrations not logged → **must be estimated** if feeds are active.                                          |
| **Flows (operation mode)** | `Vf_const`, `Vs_const`, `VGasIn_const`, `VOffGas_const`, `Va_const`, `Vb_const` | Some may appear in CSVs; if missing or zero, **must be estimated or fixed** (script currently seeds 0).              |
| **Yields**                 | `Y_Sub`, `Y_O2`, `Y_CO2`, `Y_H2O`                                               | Available in `yield_factors.csv`, but if any are missing → **estimate or use literature values**.                    |
| **Mapping parameters**     | OD → Biomass calibration factor                                                 | Critical if using OD as Biomass proxy → **estimate once for all runs**.                                              |


## Not needed or already fully known

| Parameter                | Source                   | Status        |
| ------------------------ | ------------------------ | ------------- |
| `T_celsius`              | From metadata            | fixed ≈ 30 °C |
| `V_l`, `V_g`             | Metadata / model default | known         |
| `y_O2`, `y_CO2`, `P_atm` | Air composition / model  | fixed         |
| Solver tolerances        | code defaults            | fixed         |


In [1]:
# Author: Danilo Dursoniah
# Date: 07/10/2025

import json
import math
import re
import sys
import argparse
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import pandas as pd

ASSEMBLER = None
MINIMAL_PARAMETERS = None

try:
    import importlib.util
    utils_path = Path("../bioprocess_utils.py")
    if utils_path.exists():
        spec = importlib.util.spec_from_file_location("bioprocess_utils", str(utils_path))
        bp = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(bp)  # type: ignore
        ASSEMBLER = getattr(bp, "assemble_inputs", None)
        MINIMAL_PARAMETERS = getattr(bp, "minimal_parameters", None)
except Exception as e:
    ASSEMBLER = None
    MINIMAL_PARAMETERS = None

In [2]:
import inspect

def safe_assemble_inputs(ASSEMBLER, *, params, initials, mode_name, mode_consts, t_span, t_eval, solver, meta):
    """
    Calls assemble_inputs with only the kwargs its signature accepts.
    Falls back to a manual JSON schema if ASSEMBLER is None.
    """
    if ASSEMBLER is None:
        return {
            "params": params,
            "initials": initials,
            "mode_name": mode_name,
            "mode_consts": mode_consts,
            "t_span": t_span,
            "t_eval": t_eval,
            "solver": solver,
            "meta": meta,
        }
    sig = inspect.signature(ASSEMBLER)
    full_kwargs = dict(
        params=params,
        initials=initials,
        mode_name=mode_name,
        mode_consts=mode_consts,
        t_span=t_span,
        t_eval=t_eval,
        solver=solver,
        meta=meta,
    )
    filtered = {k: v for k, v in full_kwargs.items() if k in sig.parameters}
    return ASSEMBLER(**filtered)

In [3]:
def parse_known_cli():
    ap = argparse.ArgumentParser(description="Build inputs.json-like files from AMBR datasets")
    ap.add_argument("--data-glob", type=str, default="../experimental_dataset/ambr_run*.csv",
                    help="Glob pattern to AMBR CSV files")
    ap.add_argument("--meta", type=str, default="../experimental_dataset/ambr.xlsx",
                    help="Path to metadata Excel file (optional)")
    ap.add_argument("--yields", type=str, default="../experimental_dataset/yield_factors.csv",
                    help="Path to yields CSV file (optional)")
    ap.add_argument("--out-dir", type=str, default="./",
                    help="Output directory for generated inputs JSONs")
    ap.add_argument("--od2x", type=float, default=0.4,
                    help="OD600 to Biomass (gDW/L) conversion factor")
    ap.add_argument("--mode", type=str, default="batch", choices=["batch", "fed-batch", "continuous"],
                    help="Default operation mode if flows cannot be inferred")
    return ap.parse_known_args()[0]  # for jupyter compatibility 


def load_metadata(meta_path: str) -> pd.DataFrame:
    p = Path(meta_path)
    if not p.exists():
        return pd.DataFrame()
    # first sheet by default
    try:
        df = pd.read_excel(p)
        # Normalize likely ID column
        for col in df.columns:
            if re.search(r"(reactor|vessel|ambr|tunniste|id)", str(col), re.I):
                df.rename(columns={col: "ReactorID"}, inplace=True)
        # Make ReactorID like "13", "14", etc., if it contains AMBR_XX
        if "ReactorID" in df.columns:
            df["ReactorID"] = df["ReactorID"].astype(str).str.extract(r"(\d{1,2})")
        return df
    except Exception:
        return pd.DataFrame()


def load_yields(yields_path: str) -> Dict[str, Dict[str, Any]]:
    p = Path(yields_path)
    out: Dict[str, Dict[str, Any]] = {}
    if not p.exists():
        return out
    df = pd.read_csv(p)
    # Expect columns like: name,value,unit,note
    name_col = [c for c in df.columns if re.match(r"(?i)name", c)]
    value_col = [c for c in df.columns if re.match(r"(?i)value", c)]
    unit_col = [c for c in df.columns if re.match(r"(?i)unit", c)]
    if not name_col or not value_col:
        return out
    ncol, vcol = name_col[0], value_col[0]
    ucol = unit_col[0] if unit_col else None
    for _, r in df.iterrows():
        name = str(r[ncol]).strip()
        try:
            val = float(r[vcol])
        except Exception:
            continue
        unit = str(r[ucol]).strip() if ucol else ""
        out[name] = {"values": [val], "unit": unit}
    return out


def detect_reactors(columns: List[str]) -> List[str]:
    # Detect suffix "_<id>" at end of column names; return sorted unique IDs as strings
    ids = set()
    for c in columns:
        m = re.search(r"_(\d{1,2})$", str(c))
        if m:
            ids.add(m.group(1))
    return sorted(ids, key=lambda x: int(x))


def unit_safe(val: Optional[float], default: float) -> float:
    try:
        if val is None or (isinstance(val, float) and math.isnan(val)):
            return default
        return float(val)
    except Exception:
        return default


def infer_flows(df: pd.DataFrame, rid: str) -> Dict[str, float]:
    """
    Try to infer constant flows (L/min) from columns.
    Returns dict for Vf_const, Vs_const, VGasIn_const, VOffGas_const, Va_const, Vb_const.
    If not found, zero.
    """
    # Search patterns per reactor
    # Many AMBR exports include "Air_flow_<rid>" in sccm (cm3/min) or "Flow_<rid>" in mL/min
    patterns = {
        "air": rf"(air.?flow|gas.?in|aeration).*_{rid}$",
        "offgas": rf"(off.?gas|exhaust).*_{rid}$",
        "feed": rf"(feed.?1?.?flow|pump.?1).*_{rid}$",
        "base": rf"(base|naoh).*_{rid}$",
        "acid": rf"(acid|hcl).*_{rid}$",
        "sampling": rf"(sampling|take.?out|withdraw).*_{rid}$",
    }
    means = {k: None for k in patterns.keys()}
    for k, pat in patterns.items():
        cols = [c for c in df.columns if re.search(pat, str(c), re.I)]
        if cols:
            v = df[cols[0]].astype(float)
            # Heuristic: ignore zeros, take median of positive entries
            vp = v[v > 0]
            means[k] = float(vp.median()) if not vp.empty else float(v.mean())

    def to_L_per_min(val, colname: str) -> float:
        if val is None or math.isnan(val):
            return 0.0
        # crude unit detection
        if re.search(r"(sccm|cm3.?/min|ccm)", colname, re.I):
            return float(val) * 1e-3  # 1 cm3/min = 1e-3 L/min
        if re.search(r"(mL.?/min)", colname, re.I):
            return float(val) * 1e-3
        if re.search(r"(L.?/min)", colname, re.I):
            return float(val)
        # fall back: assume mL/min
        return float(val) * 1e-3

    # Map back to first matched column for unit guess
    def first_col(pat: str) -> Optional[str]:
        for c in df.columns:
            if re.search(pat, str(c), re.I):
                return str(c)
        return None

    Vf = to_L_per_min(means["feed"], first_col(patterns["feed"]) or "")
    Vs = to_L_per_min(means["sampling"], first_col(patterns["sampling"]) or "")
    Vgi = to_L_per_min(means["air"], first_col(patterns["air"]) or "")
    Vog = to_L_per_min(means["offgas"], first_col(patterns["offgas"]) or "")
    Va = to_L_per_min(means["acid"], first_col(patterns["acid"]) or "")
    Vb = to_L_per_min(means["base"], first_col(patterns["base"]) or "")

    return dict(Vf_const=Vf, Vs_const=Vs, VGasIn_const=Vgi, VOffGas_const=Vog, Va_const=Va, Vb_const=Vb)


def default_params(yields_map: Dict[str, Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    # Start from project defaults if available
    if MINIMAL_PARAMETERS is not None:
        params = MINIMAL_PARAMETERS()
    else:
        params = {}

    def set_param(name: str, value: float, unit: str):
        params[name] = {"values": [float(value)], "unit": unit}

    # Core kinetics (defaults in 1/min)
    set_param("mu_max", 0.5 / 60.0, "1/min")          # plausible seed
    set_param("K_subs", 0.1, "g/L")
    set_param("K_L_a", 2.0, "1/min")                  # ≈ 120 1/h

    # Temperature
    set_param("T_celsius", 30.0, "degC")

    # Volumes (can be overwritten by metadata later)
    set_param("V_l", 0.2, "L")
    set_param("V_g", 0.105, "L")

    # Gas composition (as fractions) and pressure if used in utils
    set_param("y_O2", 0.21, "1")
    set_param("y_CO2", 0.0004, "1")
    set_param("P_atm", 1.0, "atm")

    # Feed concentrations — set to zeros by default; user may override later
    for sp in ["Sub", "Na", "NH4", "P", "S", "K", "Mg", "Ca", "Cl", "Co", "Cu", "Fe", "Mo", "Ni", "Zn"]:
        set_param(f"C_{sp}_f", 0.0, "mol/L")

    # Acid/base feed concentrations (if used by model)
    set_param("C_Na_b", 0.0, "mol/L")
    set_param("C_H_a", 0.0, "mol/L")

    # Yields from CSV (if provided)
    for k, v in yields_map.items():
        params[k] = {"values": list(map(float, v.get("values", [v]))), "unit": v.get("unit", "")}

    return params


def default_initials(OD0: Optional[float], od2x: float, meta_row: Optional[pd.Series]) -> Dict[str, Dict[str, Any]]:
    initials: Dict[str, Dict[str, Any]] = {}

    def set_ic(name: str, value: float, unit: str):
        initials[name] = {"values": [float(value)], "unit": unit}

    # Biomass from OD (seed if missing)
    if OD0 is None or (isinstance(OD0, float) and math.isnan(OD0)):
        # SEED: plausible 0.05 g/L when no OD at t=0
        set_ic("Biomass", 0.05, "g/L")  # <-- seeded
    else:
        set_ic("Biomass", float(OD0) * od2x, "g/L")

    # Substrate (g/L) — SEED: 10 g/L if unknown
    set_ic("Sub", 10.0, "g/L")  # <-- seeded; update if known in metadata

    # Dissolved gases (mol/L) — plausible seeds near air equilibrium at 30°C
    set_ic("O2_l", 2.5e-4, "mol/L")   # ~8 mg/L
    set_ic("CO2_l", 1.5e-5, "mol/L")

    # Gas phase (model-specific; many implementations keep gases as conc. in phases)
    set_ic("O2_g", 0.21, "mol frac")   # abstracted; model may transform to partial pressure
    set_ic("CO2_g", 4e-4, "mol frac")

    # Electrolytes & traces — start near zero; will be fed by C_*_f if needed
    for sp in ["Na", "NH4", "P", "S", "K", "Mg", "Ca", "Cl", "Co", "Cu", "Fe", "Mo", "Ni", "Zn", "H", "OH"]:
        set_ic(sp, 0.0, "mol/L")

    # Volumes as states if modeled separately; otherwise params handle them
    # set_ic("V_l", 0.2, "L")

    return initials


def build_one_inputs(
    run_name: str,
    rid: str,
    df: pd.DataFrame,
    meta_df: pd.DataFrame,
    yields_map: Dict[str, Dict[str, Any]],
    mode_name: str,
    od2x: float
) -> Dict[str, Any]:
    # time column
    tcol = [c for c in df.columns if re.match(r"(?i)time", str(c))]
    if not tcol:
        raise ValueError(f"No Time column in dataset for reactor {rid}")
    t_hours = pd.to_numeric(df[tcol[0]], errors="coerce").fillna(method="ffill").fillna(0.0).values
    t_minutes = (t_hours * 60.0).tolist()

    # reactor-specific OD column
    od_cols = [c for c in df.columns if re.search(rf"(OD|Optical.?density).*_{rid}$", str(c), re.I)]
    OD0 = None
    if od_cols:
        OD0 = pd.to_numeric(df[od_cols[0]], errors="coerce").iloc[0]

    # metadata row for reactor
    meta_row = None
    if not meta_df.empty and "ReactorID" in meta_df.columns:
        sub = meta_df[meta_df["ReactorID"] == rid]
        if not sub.empty:
            meta_row = sub.iloc[0]

    # params & initials
    params = default_params(yields_map)
    initials = default_initials(OD0, od2x, meta_row)

    # override volumes & temperature from metadata when available
    if meta_row is not None:
        # Liquid volume
        for c in meta_row.index:
            if re.search(r"(liquid|work).*(vol|volume)", str(c), re.I):
                try:
                    v = float(meta_row[c])
                    params["V_l"]["values"] = [v]
                except Exception:
                    pass
            if re.search(r"(temp|temperature)", str(c), re.I):
                try:
                    params["T_celsius"]["values"] = [float(meta_row[c])]
                except Exception:
                    pass

    # Operation mode constants
    mode_consts = infer_flows(df, rid)

    # choose mode: if all flows zero, fall back to requested default
    if all(abs(v) < 1e-12 for v in mode_consts.values()):
        use_mode = mode_name
    else:
        # if feed present -> fed-batch; if gas in present -> leave as is
        use_mode = "fed-batch" if mode_consts.get("Vf_const", 0.0) > 0 else mode_name

    # solver defaults
    solver = {"method": "LSODA", "rtol": 1e-6, "atol": 1e-9}

    # t_span & t_eval
    t0, t1 = float(min(t_minutes)), float(max(t_minutes))
    t_span = [t0, t1]
    t_eval = t_minutes  # use measurement grid

    inputs = safe_assemble_inputs(
    ASSEMBLER,
    params=params,
    initials=initials,
    mode_name=use_mode,
    mode_consts=mode_consts,
    t_span=t_span,
    t_eval=t_eval,
    solver={"method": "LSODA", "rtol": 1e-6, "atol": 1e-9},
    meta={"run": run_name, "reactor": rid},
    )

    return inputs


def split_by_reactor(df: pd.DataFrame) -> List[str]:
    return detect_reactors(df.columns.tolist())


def build_all(
    data_glob: str,
    meta_path: str,
    yields_path: str,
    out_dir: str,
    default_mode: str = "batch",
    od2x: float = 0.4
) -> List[Path]:
    out_paths: List[Path] = []
    meta_df = load_metadata(meta_path)
    yields_map = load_yields(yields_path)

    for csv_path in sorted(Path().glob(data_glob) if any(ch in data_glob for ch in "*?[]") else [Path(data_glob)]):
        if not csv_path.exists() or not csv_path.suffix.lower() == ".csv":
            continue
        run_name = csv_path.stem  # e.g., ambr_run1_140323_13-18
        df = pd.read_csv(csv_path)

        # Ensure numeric time
        tcol = [c for c in df.columns if re.match(r"(?i)time", str(c))]
        if not tcol:
            print(f"[WARN] No Time column in {csv_path.name}; skipping")
            continue

        reactor_ids = split_by_reactor(df)
        if not reactor_ids:
            # Single-reactor export without suffix — assume '1'
            reactor_ids = ["1"]

        out_dir_p = Path(out_dir)
        out_dir_p.mkdir(parents=True, exist_ok=True)

        for rid in reactor_ids:
            inputs = build_one_inputs(
                run_name=run_name,
                rid=rid,
                df=df,
                meta_df=meta_df,
                yields_map=yields_map,
                mode_name=default_mode,
                od2x=od2x
            )
            out_path = out_dir_p / f"{run_name}__R{rid}__inputs.json"
            with open(out_path, "w") as f:
                json.dump(inputs, f, indent=2)
            out_paths.append(out_path)
            print(f"[OK] Wrote {out_path}")

    return out_paths


In [4]:
args = parse_known_cli()
paths = build_all(
    data_glob=args.data_glob,
    meta_path=args.meta,
    yields_path=args.yields,
    out_dir=args.out_dir,
    default_mode=args.mode,
    od2x=args.od2x
)

[OK] Wrote ambr_run1_140323_13-18__R1__inputs.json
[OK] Wrote ambr_run1_140323_19-24__R1__inputs.json
[OK] Wrote ambr_run2_030523_11-16__R1__inputs.json
[OK] Wrote ambr_run2_030523_17-22__R1__inputs.json
[OK] Wrote ambr_run2_030523_23-24__R1__inputs.json
[OK] Wrote ambr_run2_030523_5-10__R1__inputs.json


/tmp/ipykernel_14912/2055708648.py:225: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  t_hours = pd.to_numeric(df[tcol[0]], errors="coerce").fillna(method="ffill").fillna(0.0).values
/tmp/ipykernel_14912/2055708648.py:225: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  t_hours = pd.to_numeric(df[tcol[0]], errors="coerce").fillna(method="ffill").fillna(0.0).values
/tmp/ipykernel_14912/2055708648.py:225: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  t_hours = pd.to_numeric(df[tcol[0]], errors="coerce").fillna(method="ffill").fillna(0.0).values
/tmp/ipykernel_14912/2055708648.py:225: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  t_hours = pd

### ambr_run1_140323_13-18.csv  
→ **to estimate:**  
[mu_max, K_subs, Biomass_0, Sub_0, OD_to_Biomass_factor, K_L_a, O2_l_0, CO2_l_0, (C_Na_b, C_H_a if titration modeled)]  

→ **observables (present in columns):**  
[OD, DO, OUR, CER, Offgas_CO2%, Gas_in_flow, Off_gas_flow, Temperature, pH, Liquid_volume, (Acid_flow if present), (Base_flow if present)]


### ambr_run1_140323_19-24.csv  
→ **to estimate:**  
[mu_max, K_subs, Biomass_0, Sub_0, OD_to_Biomass_factor, K_L_a, O2_l_0, CO2_l_0, (C_Na_b, C_H_a if titration modeled)]  

→ **observables:**  
[OD, DO, OUR, CER, Offgas_CO2%, Gas_in_flow, Off_gas_flow, Temperature, pH, Liquid_volume, (Acid_flow if present), (Base_flow if present)]


### ambr_run2_030523_5-10.csv  
→ **to estimate:**  
[mu_max, K_subs, Biomass_0, Sub_0, OD_to_Biomass_factor, K_L_a, O2_l_0, CO2_l_0, (C_Na_b, C_H_a if titration modeled)]  

→ **observables:**  
[OD, DO, OUR, CER, Offgas_CO2%, Gas_in_flow, Off_gas_flow, Temperature, pH, Liquid_volume, (Acid_flow if present), (Base_flow if present)]


### ambr_run2_030523_11-16.csv  
→ **to estimate:**  
[mu_max, K_subs, Biomass_0, Sub_0, OD_to_Biomass_factor, K_L_a, O2_l_0, CO2_l_0, (C_Na_b, C_H_a if titration modeled)]  

→ **observables:**  
[OD, DO, OUR, CER, Offgas_CO2%, Gas_in_flow, Off_gas_flow, Temperature, pH, Liquid_volume, (Acid_flow if present), (Base_flow if present)]


### ambr_run2_030523_17-22.csv  
→ **to estimate:**  
[mu_max, K_subs, Biomass_0, Sub_0, OD_to_Biomass_factor, K_L_a, O2_l_0, CO2_l_0, (C_Na_b, C_H_a if titration modeled)]  

→ **observables:**  
[OD, DO, OUR, CER, Offgas_CO2%, Gas_in_flow, Off_gas_flow, Temperature, pH, Liquid_volume, (Acid_flow if present), (Base_flow if present)]


### ambr_run2_030523_23-24.csv  
→ **to estimate:**  
[mu_max, K_subs, Biomass_0, Sub_0, OD_to_Biomass_factor, K_L_a, O2_l_0, CO2_l_0, (C_Na_b, C_H_a if titration modeled)]  

→ **observables:**  
[OD, DO, OUR, CER, Offgas_CO2%, Gas_in_flow, Off_gas_flow, Temperature, pH, Liquid_volume, (Acid_flow if present), (Base_flow if present)]


---

### Notes specific to **batch mode**
- **Feed_flow** ≡ 0 → exclude from observables; drop `C_Sub_f` and don’t estimate `Vf_const`.  
- Handle **sampling** as discrete volume removals inferred from `Liquid_volume` steps → exclude continuous `Sampling_flow`.  
- If `Gas_in_flow` or `Off_gas_flow` columns are missing → fix to known constants or rely on `K_L_a` alone (avoid co-estimating).  
- If **pH control** is absent (acid/base flows zero or missing) → drop `(C_Na_b, C_H_a)` and exclude Acid/Base from observables.  
